In [60]:
import os
import pandas as pd
# 自动定位项目根目录：notebook 位于 notes/ 下，向上一级即为项目根
# 如果 Jupyter 已经从项目根目录启动，则无需跳转
# 后续 'data/ShanghaiPM_Training.csv' 等相对路径即可在任何环境中正确解析
cwd = os.getcwd()
if os.path.basename(cwd) == 'notes':
    os.chdir('..')
print('当前工作目录:', os.getcwd())

当前工作目录: D:\github\prediction


In [61]:
df_raw = pd.read_csv('data/ShanghaiPM_Training.csv', na_values='NA')
print('训练集形状:', df_raw.shape)
df_raw.info()

训练集形状: (52183, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52183 entries, 0 to 52182
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   No             52183 non-null  int64  
 1   year           52183 non-null  int64  
 2   month          52183 non-null  int64  
 3   day            52183 non-null  int64  
 4   hour           52183 non-null  int64  
 5   season         52183 non-null  int64  
 6   PM_Jingan      24305 non-null  float64
 7   PM_US Post     33663 non-null  float64
 8   PM_Xuhui       24792 non-null  float64
 9   DEWP           52170 non-null  float64
 10  HUMI           52170 non-null  float64
 11  PRES           52155 non-null  float64
 12  TEMP           52170 non-null  float64
 13  cbwd           52171 non-null  object 
 14  Iws            52171 non-null  float64
 15  precipitation  48318 non-null  float64
 16  Iprec          48318 non-null  float64
dtypes: float64(10), int64(6), objec

In [62]:
# pd.to_datetime() 的作用：
#   把包含日期/时间信息的列（或列的组合）转换成 Python 的 Datetime 类型
#   这里我们传入一个字典，告诉 pandas：
#     把 year 列当作"年"，month 列当作"月"，day 列当作"日"，hour 列当作"时"
df_raw['datetime'] = pd.to_datetime(
    {'year': df_raw['year'], 'month': df_raw['month'], 'day': df_raw['day'], 'hour': df_raw['hour']}
)

# df.set_index() 把指定的列变成 DataFrame 的行索引（类似 Excel 最左边那列行号）
df_raw = df_raw.set_index('datetime')

# df.sort_index() 按行索引排序
df_raw = df_raw.sort_index()

# df.drop() 删除不需要的列
# columns=[...] 指定要删除的列名列表
# errors='ignore' 表示如果某列不存在也不报错（容错处理）

pm_cols = ['PM_Jingan', 'PM_US Post', 'PM_Xuhui']
df_raw['pm_ave'] = df_raw[pm_cols].mean(axis=1)
df_raw = df_raw.dropna(subset=['pm_ave'])
df_raw.head()

,No,year,month,day,hour,season,PM_Jingan,PM_US Post,PM_Xuhui,DEWP,HUMI,PRES,TEMP,cbwd,Iws,precipitation,Iprec,pm_ave
datetime,,,,,,,,,,,,,,,,,,
2011-12-28 18:00:00,17443,2011,12,28,18,4,NaN,36.0,NaN,4.0,62.00,1027.1,11.0,NE,8.0,0.0,0.0,36.0
2011-12-28 19:00:00,17444,2011,12,28,19,4,NaN,41.0,NaN,4.0,62.00,1027.1,11.0,NE,9.0,0.0,0.0,41.0
2011-12-28 20:00:00,17445,2011,12,28,20,4,NaN,44.0,NaN,5.0,71.07,1028.1,10.0,NE,11.0,0.0,0.0,44.0
2011-12-28 21:00:00,17446,2011,12,28,21,4,NaN,40.0,NaN,5.0,71.07,1028.1,10.0,NE,13.0,0.0,0.0,40.0
2011-12-28 22:00:00,17447,2011,12,28,22,4,NaN,25.0,NaN,6.0,76.18,1028.1,10.0,NE,15.0,0.0,0.0,25.0


In [63]:
KEEP_COLS = [
    'pm_ave',       # 徐汇站 PM2.5 浓度 (ug/m3)
    'TEMP',           # 气温 (摄氏度)
    'HUMI',           # 相对湿度 (%)
]

# df[[...]] 通过列名列表来筛选子集
# .copy() 创建一个独立的副本——这是好习惯，防止后续操作意外影响原始数据
df = df_raw[KEEP_COLS].copy()

print('最终数据形状:', df.shape)
print('列名:', list(df.columns))
print('索引类型:', type(df.index))  # 应该是 DatetimeIndex
df.head(10)

最终数据形状: (34394, 3)
列名: ['pm_ave', 'TEMP', 'HUMI']
索引类型: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>


,pm_ave,TEMP,HUMI
datetime,,,
2011-12-28 18:00:00,36.0,11.0,62.00
2011-12-28 19:00:00,41.0,11.0,62.00
2011-12-28 20:00:00,44.0,10.0,71.07
2011-12-28 21:00:00,40.0,10.0,71.07
2011-12-28 22:00:00,25.0,10.0,76.18
2011-12-28 23:00:00,28.0,10.0,76.18
2011-12-29 00:00:00,34.0,10.0,76.18
2011-12-29 01:00:00,25.0,9.0,76.01
2011-12-29 02:00:00,27.0,9.0,76.01


In [64]:
# df.isna() 返回一个布尔型 DataFrame（True 表示该位置是缺失值 NaN）
# .sum() 对每列求和时，True 自动按 1 计数 → 得到每列的缺失数量
print('各列缺失数量:')
print(df.isna().sum())

# df.describe() 输出每一列的统计量：
#   count（非空个数）, mean（均值）, std（标准差）,
#   min（最小值）, 25%, 50%, 75%, max（最大值）
# .round(2) 把小数四舍五入到 2 位，让输出更整洁
print('\n统计摘要:')
df.describe().round(2)

各列缺失数量:
pm_ave    0
TEMP      5
HUMI      5
dtype: int64

统计摘要:


,pm_ave,TEMP,HUMI
count,34394.00,34389.00,34389.00
mean,54.40,17.63,69.93
std,44.12,9.17,17.77
min,1.00,-4.00,13.09
25%,25.67,10.00,58.29
50%,42.00,19.00,72.96
75%,69.00,25.00,83.60
max,730.00,41.00,100.00
